# Advanced Analytics for a Better World — Lecture 2\n\n[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gromicho/teaching/blob/main/courses/aabw/notebooks/lecture-2/elizabeth-facility-location.ipynb) [![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/gromicho/teaching/main?urlpath=tree/courses/aabw/notebooks/lecture-2/elizabeth-facility-location.ipynb)\n\n> Original Lecture 2 notebook, retained with its author attribution and content.\n

# Data Science Essentials: Applied Optimization

Joaquim Gromicho, 2021

This notebook is part of the module Applied Optimization of the Analytics Academy's Data Science Essentials.

---
 > During this course we make use of Jupyter notebooks hosted by [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb).
  Notebooks deployed on `colab` require neither python nor other dependencies to be installed on your own machine, you only need a browser (preferably `chrome`) and you may also need a google account if you want to execute them.

---

Let us suppose now that Caroline is so successful that she considers opening a number of distribution centers to support scaling up her production.

Taking into account her costumers $J$ and a set $I$ of possible locations for her new distribution centers she estimates the costs $c_j$ of opening at $j \in J$ and $d_{ij}$ of serving $i \in I$ from $j \in J$.

Her decision variables are:
     
$$
   x_j = \left\{
     \begin{array}{lr}
       1 & \mbox{center } j \mbox{ is built}\\
       0 & \mbox{otherwise}\\
     \end{array}
   \right.
       \mbox{ and }
   y_{ij} = \left\{
     \begin{array}{lr}
       1 & \mbox{customer } i \mbox{ is served at } j\\
       0 & \mbox{otherwise}\\
     \end{array}
   \right.
$$

Minimizing the number of bins used provided that each item goes in a bin and the sum of the sizes $s_i$ of items in the same bin do not exceed the capacity $c$ is:
$$
\begin{array}{rrcll}
\min    & \sum_{j\in J} c_jx_j + \sum_{i \in I, j \in J} d_{ij}y_{ij}\\
s.t.    & \sum_{j\in J} y_{ij}     & =    & 1     & \forall i \in I \\
        & \sum_{i\in I} y_{ij}     & \leq & n x_j & \forall j \in J \\
        & y_{ij} \in \{0,1\}       &      &       & \forall i \in I, j \in J \\
        & x_j \in \{0,1\}          &      &       & \forall j \in J \\
\end{array}
$$
       
The disaggregated model which is claimed to be stronger is:
$$
\begin{array}{rrcll}
\min    & \sum_{j\in J} c_jx_j + \sum_{i \in I, j \in J} d_{ij}y_{ij}\\
s.t.    & \sum_{j\in J} y_{ij}     & =    & 1     & \forall i \in I \\
        &               y_{ij}     & \leq & x_j   & \forall i \in I, j \in J \\
        & y_{ij} \in \{0,1\}       &      &       & \forall i \in I, j \in J \\
        & x_j \in \{0,1\}          &      &       & \forall j \in J \\
\end{array}
$$

In [ ]:
%matplotlib inline


In [ ]:
# A lightweight, open-source solver baseline for Colab and Binder.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('pyomo') is None or importlib.util.find_spec('highspy') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'pyomo', 'highspy'])

import pyomo.environ as pyo

solver_name = 'appsi_highs'
solver = pyo.SolverFactory(solver_name)
if not solver.available(exception_flag=False):
    raise RuntimeError('HiGHS is unavailable. Restart the runtime and run this cell again.')


In [ ]:
def FacilityLocationWeak( installation, service, solver=solver_name ):
    from pyomo.environ import ConcreteModel, Var, Objective, Constraint, NonNegativeReals, Binary, minimize

    model = ConcreteModel("Facility location")
    model.nofFacilities = len( installation )
    model.nofCustomers  = len( service )
    model.facilities = range( model.nofFacilities )
    model.customers  = range( model.nofCustomers )

    model.x = Var( model.facilities, within=Binary )
    model.y = Var( model.customers, model.facilities, within=Binary )

    model.obj = Objective ( expr = sum([installation[j]*model.x[j] for j in model.facilities])                          \
                                 + sum([service[i][j]*model.y[i,j] for i in model.customers for j in model.facilities]) \
                          , sense=minimize)

    def ServeIfOpen( model, j ):
        return sum([model.y[i,j] for i in model.customers]) <= model.nofCustomers*model.x[j]

    def ChooseOneFacility( model, i ):
        return sum([model.y[i,j] for j in model.facilities]) == 1

    model.serve  = Constraint( model.facilities, rule=ServeIfOpen )
    model.choose = Constraint( model.customers, rule=ChooseOneFacility )

    from pyomo.opt import SolverFactory
    %time results = SolverFactory(solver).solve(model)

    X = [ model.x[j].value >= .5 for j in model.facilities ]
    Y = [ [ model.y[i,j].value >= .5 for j in model.facilities ] for i in model.customers ]

    return X, Y, model.obj.expr()

def FacilityLocationStrong( installation, service, solver=solver_name ):
    from pyomo.environ import ConcreteModel, Var, Objective, Constraint, NonNegativeReals, Binary, minimize

    model = ConcreteModel("Gina's facility location")
    model.nofFacilities = len( installation )
    model.nofCustomers  = len( service )
    model.facilities = range( model.nofFacilities )
    model.customers  = range( model.nofCustomers )

    model.x = Var( model.facilities, within=Binary )
    model.y = Var( model.customers, model.facilities, within=Binary )

    model.obj = Objective ( expr = sum([installation[j]*model.x[j] for j in model.facilities])                          \
                                 + sum([service[i][j]*model.y[i,j] for i in model.customers for j in model.facilities]) \
                          , sense=minimize)

    def ServeIfOpen( model, i, j ):
        return model.y[i,j] <= model.x[j]

    def ChooseOneFacility( model, i ):
        return sum([model.y[i,j] for j in model.facilities]) == 1

    model.serve  = Constraint( model.customers, model.facilities, rule=ServeIfOpen )
    model.choose = Constraint( model.customers, rule=ChooseOneFacility )

    from pyomo.opt import SolverFactory
    %time results = SolverFactory(solver).solve(model,tee=True)

    X = [ model.x[j].value >= .5 for j in model.facilities ]
    Y = [ [ model.y[i,j].value >= .5 for j in model.facilities ] for i in model.customers ]

    return X, Y, model.obj.expr()

In [ ]:
def GenerateFacilityLocationInstance( nofFacilities, nofCustumers ):
    facilities = range(nofFacilities)
    customers = range(nofCustumers)
    import numpy as np
    xC = np.random.randint( 0, 100, nofCustumers )
    yC = np.random.randint( 0, 100, nofCustumers )
    xF = np.random.randint( 0, 100, nofFacilities )
    yF = np.random.randint( 0, 100, nofFacilities )

    installation = np.random.randint( 100, 200, nofFacilities )

    dist = lambda i,j : ((xC[i]-xF[j])**2 + (yC[i]-yF[j])**2)

    service = [ [ dist(i,j) for j in facilities ] for i in customers ]

    return installation, service, xC, yC, xF, yF

In [ ]:
def ShowFacilityLocation( xC, yC, xF, yF, X=[], Y=[], cost=None ):
    import matplotlib.pyplot as plt
    [ plt.plot( [xC[i],xF[j]], [yC[i],yF[j]], 'g-' ) for j in range(len(X)) if X[j] for i in range(len(Y)) if Y[i][j] ]
    plt.plot( xC,yC, 'o' )
    plt.plot( xF,yF, 's' )
    if not cost is None:
        plt.title( str(cost) )
    plt.show()

In [ ]:
installation, service, xC, yC, xF, yF = GenerateFacilityLocationInstance(30,500)

In [ ]:
ShowFacilityLocation( xC, yC, xF, yF )

In [ ]:
solver = solver_name
#%time ShowFacilityLocation( xC, yC, xF, yF, *FacilityLocationWeak( installation, service, solver=solver ) )

In [ ]:
%time ShowFacilityLocation( xC, yC, xF, yF, *FacilityLocationStrong( installation, service, solver=solver_name ) )

# The solver may aliviate the problem

`glpk` is ideal to test the strength of models, since it does not modify the instances very much. `cbc` is already a bit more clever...

In [ ]:
solver = solver_name
%time ShowFacilityLocation( xC, yC, xF, yF, *FacilityLocationWeak( installation, service, solver=solver ) )

In [ ]:
%time ShowFacilityLocation( xC, yC, xF, yF, *FacilityLocationStrong( installation, service, solver=solver ) )

The commercial solvers are very good at disagregating. If we keep our instances small we can solve them with the free community editions of `gurobi`, `cplex` and `xpress`.

### Optional commercial solvers

Gurobi, CPLEX, and Xpress require their own licences and local configuration. They are intentionally not installed by this course notebook.


The optional commercial-solver comparison was removed from the standard execution path. The open-source HiGHS solver above is the supported course baseline.
